# <img style="float: left; padding-right: 10px; width: 100px" src="https://raw.githubusercontent.com/sbildea/SEED/main/imagini/unitbv_mie.png"> <center>Știința Explorării și Exploatării Datelor - SEED1</center>


##    <center>Extra 2: Lucrul cu dataframe-uri. Analiza exploratorie</center>
**Universitatea Transilvania**<br>
**SIIPA, Semestrul I<br>
**Instructor:** Conf. dr. Teodor &Scedil;tefan B&icirc;ldea<br>
**Contact:** teodor.bildea@unitbv.ro<br>

### Cuprins
0. Recapitulare laborator 1
1. Citirea fisierelor 
2. Combinarea mai multor dataframe-uri
3. Lucrul cu dataframe-uri
4. Vizualizarea datelor: matplotlib, plotly

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sbildea/SEED/blob/main/cursuri/02-explorare-vizualizare/extra-02-dataframe.ipynb)

In [ ]:
# Configurare curs SEED: rulați această celulă prima.
# În Colab descarcă materialele cursului și instalează pachetele lipsă (aprox. un minut).
import os, sys
if "google.colab" in sys.modules:
    if not os.path.exists("/content/SEED"):
        !git clone -q --depth 1 https://github.com/sbildea/SEED.git /content/SEED
        !pip install -q -r /content/SEED/requirements.txt
    %cd /content/SEED/cursuri/02-explorare-vizualizare
sys.path.insert(0, os.path.abspath("../.."))

# Recapitulare laborator 1

- indentare în Python

- lucrul  cu celule în Jupyter

- instrucțiuni de bază: interogare - if-then-else; bucle - for, while

- liste și operații cu liste

- dicționare și operații cu dicționare

- folosirea pachetelor open-source: import

Importuri necesare

In [3]:
# libraria pentru lucrul tabelar
import pandas as pd

# libraria pentru functii matematice
import numpy as np

`INFO POINT` Resurse open-source care nu sunt instalate se pot instala fie temporar, la nivel de Jupyter Notebook, fie persistent, în Anaconda (din interfața Anaconda sau din terminalul Anaconda). Voi exemplefica cănd va fi cazul.

## tipuri de date

>1 .csv
- se foloseste metoda pandas `read_csv` pt citirea unui fisier .csv
- daca sunt mai multe se citesc pe rand, si din documentatie se afla cheile/id-uri pe care le putem folosi pentru a crea un singur tabel mare in urma unei secvente de instructiuni 'merge' despre care vom vorbi mai jos

>2 .xlsx
- se foloseste metoda pandas `read_excel` si se specifica cel putin worksheetul

>3 Alte formate de date: sqlite
- se foloseste libraria `sqlite3` - este un import mai special, vom vedea mai jos detalii

>4 Date tip .json
- se foloseste metoda `read_json` din pandas


# Niste configurari

### MOTTO:  <i>Încurcate sunt căile ...</i>

Căile către directoare se specifică în Windows folosind caracterul `\`, care dă multă bătaie de cap în programare. Acest caracter este specificat fie ca `\\`, fie ca `/`, fie se poate specifica direct calea folosind prefixul `r` ca mai jos.

In [4]:
# calea către seturile de date: descarca() aduce arhiva setului (doar prima dată)
# și întoarce folderul în care a fost dezarhivată
from seed.date import descarca
input_path = descarca("date-curs")
input_path

Se poate folosi calea relativă către date. De exemplu, dacă la același nivel cu acest notebook există directorul `date` atunci se poate specifica:

## Citirea datelor

Nu mai trebuie schimbată nicio cale: `descarca()` aduce datele automat, la fel în Colab și pe calculatorul vostru.

## Date tip .csv

Pentru proiectele cu mai multe fișiere căutăm prima dată fișierele și construim dicționare de dataframe-uri.

`INFO POINT` <b><i>try-except</i></b> este cuplul folosit pentru a 'prinde' eventuale erori în cod - dacă apare o eroare la executarea codului de pe ramura <b>try</b>, execuția se întrerupe și codul 'sare' la ramura <b>except</b>

In [6]:
# import pachet functii pentru lucrul cu sistemul de operare, nume de directoare si nume de fisiere
import glob
import os

# cale catre date
path = os.path.join(input_path ,"csv/*.csv")

# declarare si initializare dictionar gol
dict_df = {}

# parcurgeree in bucla for a liste ce contine fisiere din directorul specifica in cale
for fname in glob.glob(path):
    
    #print(fname)
    
    # try-except este cuplul folosit pentru a 'prinde' eventuale erori in cod - daca apare o eroare
    # la executare codului de pe ramura 'try', executia se intrerupe si se 'sare' la ramura 'except'
    try:
        # citire fisier csv si adaugare in dictionar cu cheia = nume fara cale si terminatie
        dict_df[os.path.basename(fname).replace(".csv","")] = \
                            pd.read_csv(fname)
    
    except:
        print('eroare la citire fisier {0}', fname)

    
# verificam citirea
for key in dict_df.keys():
    print(key, dict_df[key].shape)

audi (10668, 9)
bmw (10781, 9)
cclass (3899, 7)
focus (5454, 7)
ford (17965, 9)
hyundi (4860, 9)
merc (13119, 9)
skoda (6267, 9)
toyota (6738, 9)
unclean cclass (4006, 11)
unclean focus (5604, 11)
vauxhall (13632, 9)
vw (15157, 9)


## Date tip .xlsx

Păstrez structura de mai sus în caz că ar putea fi mai multe fișiere input în format xlsx.

In [7]:
import glob
path = os.path.join(input_path, "xlsx", "*.xlsx")

dict_df = {}
for fname in glob.glob(path):
    #print(fname)
    try:
        # citire fisier csv si adaugare in dictionar cu cheia = nume fara cale si terminatie
        dict_df[os.path.basename(fname).replace(".csv","")] = \
                            pd.read_excel(fname, sheet_name = "Sheet1")
    
    except:
        print('eroare la citire fisier {0}', fname)

    
# verificam citirea
for key in dict_df.keys():
    print(key, dict_df[key].shape)

Kaggle_Sirio_Libanes_ICU_Prediction.xlsx (1925, 231)


## Date tip sqlite3

Citirea se fac atipic. Pandas nu are o metodă dedicată. Se folosește modului `sqlite3`.

In [8]:
import sqlite3

# conectarea la db
conn = sqlite3.connect(descarca("lab-set8") / "database.sqlite")

# numele tabelelor
df_tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;", conn)
display(df_tables)

# lista cu dataframe-uri
dataframes = {}
for _,row in df_tables.iterrows():

    print('extragere date tabel ', row['name'])

    df_temp = pd.read_sql_query("SELECT * FROM " + row['name'], conn)

    dataframes[row['name']] = df_temp



,name
0,Country
1,League
2,Match
3,Player
4,Player_Attributes
5,Team
6,Team_Attributes
7,sqlite_sequence


extragere date tabel  Country
extragere date tabel  League
extragere date tabel  Match


extragere date tabel  Player
extragere date tabel  Player_Attributes
extragere date tabel  Team
extragere date tabel  Team_Attributes
extragere date tabel  sqlite_sequence


## Date in format .json

Veti întâlni adesea acest format, a devenit foarte popular. Aici avem un exemplu de arhivă cu un singur fișier. Toate metodele ce citire din `pandas` știu  să dezarhiveze. Am folosit această ocazie pentru a exemplifica.



In [9]:
import json
import glob
    
fname = os.path.join(input_path, "json", "winemag-data-130k-v2.zip")
dict_df = {}
#print(fname)
try:
    # citire fisier csv si adaugare in dictionar cu cheia = nume fara cale si terminatie
    dict_df[os.path.basename(fname).replace(".zip","")] = \
                        pd.read_json(fname, compression = 'zip')

except:
    print('eroare la citire fisier {0}', fname)

    
# verificam citirea
for key in dict_df.keys():
    print(key, dict_df[key].shape)


winemag-data-130k-v2 (129971, 13)


In [18]:
mydf = dict_df['winemag-data-130k-v2']
display(mydf)

,points,title,description,taster_name,taster_twitter_handle,price,designation,variety,region_1,region_2,province,country,winery
0,87,Nicosia 2013 Vulkà Bianco (Etna),"Aromas include tropical fruit, broom, brimston...",Kerin O’Keefe,@kerinokeefe,NaN,Vulkà Bianco,White Blend,Etna,None,Sicily & Sardinia,Italy,Nicosia
1,87,Quinta dos Avidagos 2011 Avidagos Red (Douro),"This is ripe and fruity, a wine that is smooth...",Roger Voss,@vossroger,15.0,Avidagos,Portuguese Red,None,None,Douro,Portugal,Quinta dos Avidagos
2,87,Rainstorm 2013 Pinot Gris (Willamette Valley),"Tart and snappy, the flavors of lime flesh and...",Paul Gregutt,@paulgwine,14.0,None,Pinot Gris,Willamette Valley,Willamette Valley,Oregon,US,Rainstorm
3,87,St. Julian 2013 Reserve Late Harvest Riesling ...,"Pineapple rind, lemon pith and orange blossom ...",Alexander Peartree,None,13.0,Reserve Late Harvest,Riesling,Lake Michigan Shore,None,Michigan,US,St. Julian
4,87,Sweet Cheeks 2012 Vintner's Reserve Wild Child...,"Much like the regular bottling from 2012, this...",Paul Gregutt,@paulgwine,65.0,Vintner's Reserve Wild Child Block,Pinot Noir,Willamette Valley,Willamette Valley,Oregon,US,Sweet Cheeks
...,...,...,...,...,...,...,...,...,...,...,...,...,...
129966,90,Dr. H. Thanisch (Erben Müller-Burggraef) 2013 ...,Notes of honeysuckle and cantaloupe sweeten th...,Anna Lee C. Iijima,None,28.0,Brauneberger Juffer-Sonnenuhr Spätlese,Riesling,None,None,Mosel,Germany,Dr. H. Thanisch (Erben Müller-Burggraef)
129967,90,Citation 2004 Pinot Noir (Oregon),Citation is given as much as a decade of bottl...,Paul Gregutt,@paulgwine,75.0,None,Pinot Noir,Oregon,Oregon Other,Oregon,US,Citation
129968,90,Domaine Gresser 2013 Kritt Gewurztraminer (Als...,Well-drained gravel soil gives this wine its c...,Roger Voss,@vossroger,30.0,Kritt,Gewürztraminer,Alsace,None,Alsace,France,Domaine Gresser
129969,90,Domaine Marcel Deiss 2012 Pinot Gris (Alsace),"A dry style of Pinot Gris, this is crisp with ...",Roger Voss,@vossroger,32.0,None,Pinot Gris,Alsace,None,Alsace,France,Domaine Marcel Deiss


## Informații utile despre un obiect dataframe
- structura (shape)
- tipurile de date ale coloanelor (dtypes)
- statistici pentru coloanele numerice (describe)
- numarul valorilor distincte (nunique) si valorile distincte (unique)
- valorile lipsa si procentul de valori bune (coverage_pct)

In [11]:
dict_df.keys()

dict_keys(['winemag-data-130k-v2'])

In [14]:
a= 1
b=2
c=a+b
c

3

In [12]:
#Exemplu pe datele de mai sus
prima_cheie = list(dict_df.keys())[0]
df = dict_df[prima_cheie]

# Operatii cu <i>un</i> dataframe

Tipul datelor

In [13]:
#dtypes

#tipuri
print("tipuri date\n", df.dtypes)
print()
print("tipuri date")
print(df.dtypes)
print()
print("tipuri date")
df.dtypes


tipuri date
 points                     int64
title                     object
description               object
taster_name               object
taster_twitter_handle     object
price                    float64
designation               object
variety                   object
region_1                  object
region_2                  object
province                  object
country                   object
winery                    object
dtype: object

tipuri date
points                     int64
title                     object
description               object
taster_name               object
taster_twitter_handle     object
price                    float64
designation               object
variety                   object
region_1                  object
region_2                  object
province                  object
country                   object
winery                    object
dtype: object

tipuri date


points                     int64
title                     object
description               object
taster_name               object
taster_twitter_handle     object
price                    float64
designation               object
variety                   object
region_1                  object
region_2                  object
province                  object
country                   object
winery                    object
dtype: object

Număr de linii și de coloane

In [12]:
#.shape

#structura
print("structura", df.shape)
print("coloane", df.shape[0])
print("linii", df.shape[1])

structura (129971, 13)
coloane 129971
linii 13


Pentru coloane care nu sunt numerice, să listăm valorile distincte.

In [13]:
numeric_cols = df.select_dtypes([np.number]).columns

other_cols = [c for c in df.columns if c not in numeric_cols]
for c in other_cols:
    print( c, df[c].nunique())

title 118840
description 119955
taster_name 19
taster_twitter_handle 15
designation 37979
variety 707
region_1 1229
region_2 17
province 425
country 43
winery 16757


In [14]:
df.info(), df.columns

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 129971 entries, 0 to 129970
Data columns (total 13 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   points                 129971 non-null  int64  
 1   title                  129971 non-null  object 
 2   description            129971 non-null  object 
 3   taster_name            103727 non-null  object 
 4   taster_twitter_handle  98758 non-null   object 
 5   price                  120975 non-null  float64
 6   designation            92506 non-null   object 
 7   variety                129970 non-null  object 
 8   region_1               108724 non-null  object 
 9   region_2               50511 non-null   object 
 10  province               129908 non-null  object 
 11  country                129908 non-null  object 
 12  winery                 129971 non-null  object 
dtypes: float64(1), int64(1), object(11)
memory usage: 12.9+ MB


(None,
 Index(['points', 'title', 'description', 'taster_name',
        'taster_twitter_handle', 'price', 'designation', 'variety', 'region_1',
        'region_2', 'province', 'country', 'winery'],
       dtype='object'))

Statistica pentru datele numerice

In [15]:
print("coloane numerice - statistici")
display(df.describe())

coloane numerice - statistici


,points,price
count,129971.000000,120975.000000
mean,88.447138,35.363389
std,3.039730,41.022218
min,80.000000,4.000000
25%,86.000000,17.000000
50%,88.000000,25.000000
75%,91.000000,42.000000
max,100.000000,3300.000000


In [16]:
# .describe()

print("coloane numerice - statistici")
display(df.describe().T)

coloane numerice - statistici


,count,mean,std,min,25%,50%,75%,max
points,129971.0,88.447138,3.039730,80.0,86.0,88.0,91.0,100.0
price,120975.0,35.363389,41.022218,4.0,17.0,25.0,42.0,3300.0


In [17]:
display(df.describe(percentiles = [0.8,0.85,0.9]).T)

,count,mean,std,min,50%,80%,85%,90%,max
points,129971.0,88.447138,3.039730,80.0,88.0,91.0,92.0,92.0,100.0
price,120975.0,35.363389,41.022218,4.0,25.0,48.0,55.0,65.0,3300.0


In [18]:
import numpy as np

# diferenta intre medie si mediana:
# sir = [3, 11, 15, 18, 5, 17, 6, 20, 1, 14, 12, 10, 7, 4, 2, 0,0,0,0]
sir = [1,2,2,2,2]
media = sum(sir)/len(sir)
mediana = np.median(sir)
media, mediana

(1.8, 2.0)

Raport valori lipsă

In [19]:
print("raport valori lipsa/ valori bune")

results = []
for c in df.columns:
    # filtram cu conditia ca valorile din coloana c sa fie lipsa si returnam numarul de linii
    missing_cnt = df[ df[c].isna() ].shape[0]
    dict_row ={"nume coloana": c,
                "valori lipsa": str(missing_cnt),
               "procent valori bune": round( 100*(1-missing_cnt/df.shape[0]),2)}
    results.append(dict_row)

display(pd.DataFrame(results))

raport valori lipsa/ valori bune

,nume coloana,valori lipsa,procent valori bune
0,points,0,100.00
1,title,0,100.00
2,description,0,100.00
3,taster_name,26244,79.81
4,taster_twitter_handle,31213,75.98
5,price,8996,93.08
6,designation,37465,71.17
7,variety,1,100.00
8,region_1,21247,83.65
9,region_2,79460,38.86


Dacă numărul coloanelor nu este foarte mare se poate folos `info` pentru a vedea un sumar referitor la găurile din date:

In [20]:
#.info
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 129971 entries, 0 to 129970
Data columns (total 13 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   points                 129971 non-null  int64  
 1   title                  129971 non-null  object 
 2   description            129971 non-null  object 
 3   taster_name            103727 non-null  object 
 4   taster_twitter_handle  98758 non-null   object 
 5   price                  120975 non-null  float64
 6   designation            92506 non-null   object 
 7   variety                129970 non-null  object 
 8   region_1               108724 non-null  object 
 9   region_2               50511 non-null   object 
 10  province               129908 non-null  object 
 11  country                129908 non-null  object 
 12  winery                 129971 non-null  object 
dtypes: float64(1), int64(1), object(11)
memory usage: 12.9+ MB




Puteți folosi modulul `ydata_profiling`;  pentru date mari e mai greoi (spre imposibil cănd tabelele conțin milioane de înregistrări și sute de coloane...); evitați calcularea corelațiilor (foarte costisitoare ca resurse și timp) ca în exemplul de mai jos.

`INFO POINT` Dacă nu aveți instalat pachetul <b>ydata_profiling</b> puteți apela la o instalare ad-hoc în acest notebook după cum urmează:
- într-o celulă nouă introduceți `! pip install ydata_profiling`


`ATENTIE` Dacă aveți un set de date mare, calculul diferitelor corelații pentru raport durează mult. Puteți inhiba calculele acestea - merită și numai restul raportului. Dar și acela poate dura extrem de mult pentru seturile mari de date. Mai multe puteți afla de pe site la <a href='https://docs.profiling.ydata.ai/latest/'>ydata_profiling</a>.


In [23]:
# ! pip install ydata_profiling

In [24]:
# import pandas as pd
# from ydata_profiling import ProfileReport

# raport_fara_corelatii = ProfileReport(df, title="Profiling Report", check_correlation=False)
# raport_fara_corelatii


In [25]:
# raport_complet = ProfileReport(df, title="Profiling Report")
# raport_complet

Numele coloanelor

In [26]:
#.columns
df.columns

Index(['points', 'title', 'description', 'taster_name',
       'taster_twitter_handle', 'price', 'designation', 'variety', 'region_1',
       'region_2', 'province', 'country', 'winery'],
      dtype='object')

## Un prim exemplu de filtrare

Imputăm valorile lipsă din coloana 'country' cu șirul de caractere `Missing`

Pentru aceasta vom accesa coloana si vom folosi metoda de umplere a valorilor lipsa - fillna(), cu parametru dat de valoare cu care se vor umple 'golurile'.

Pentru a verifica succesul operatțiunii, vom filtra dataframe-ul - vom reține doar înregitrările pentru care în coloana `country` sunt valori nule.

In [27]:
df[ ~df['country'].isna() ]

,points,title,description,taster_name,taster_twitter_handle,price,designation,variety,region_1,region_2,province,country,winery
0,87,Nicosia 2013 Vulkà Bianco (Etna),"Aromas include tropical fruit, broom, brimston...",Kerin O’Keefe,@kerinokeefe,NaN,Vulkà Bianco,White Blend,Etna,None,Sicily & Sardinia,Italy,Nicosia
1,87,Quinta dos Avidagos 2011 Avidagos Red (Douro),"This is ripe and fruity, a wine that is smooth...",Roger Voss,@vossroger,15.0,Avidagos,Portuguese Red,None,None,Douro,Portugal,Quinta dos Avidagos
2,87,Rainstorm 2013 Pinot Gris (Willamette Valley),"Tart and snappy, the flavors of lime flesh and...",Paul Gregutt,@paulgwine,14.0,None,Pinot Gris,Willamette Valley,Willamette Valley,Oregon,US,Rainstorm
3,87,St. Julian 2013 Reserve Late Harvest Riesling ...,"Pineapple rind, lemon pith and orange blossom ...",Alexander Peartree,None,13.0,Reserve Late Harvest,Riesling,Lake Michigan Shore,None,Michigan,US,St. Julian
4,87,Sweet Cheeks 2012 Vintner's Reserve Wild Child...,"Much like the regular bottling from 2012, this...",Paul Gregutt,@paulgwine,65.0,Vintner's Reserve Wild Child Block,Pinot Noir,Willamette Valley,Willamette Valley,Oregon,US,Sweet Cheeks
...,...,...,...,...,...,...,...,...,...,...,...,...,...
129966,90,Dr. H. Thanisch (Erben Müller-Burggraef) 2013 ...,Notes of honeysuckle and cantaloupe sweeten th...,Anna Lee C. Iijima,None,28.0,Brauneberger Juffer-Sonnenuhr Spätlese,Riesling,None,None,Mosel,Germany,Dr. H. Thanisch (Erben Müller-Burggraef)
129967,90,Citation 2004 Pinot Noir (Oregon),Citation is given as much as a decade of bottl...,Paul Gregutt,@paulgwine,75.0,None,Pinot Noir,Oregon,Oregon Other,Oregon,US,Citation
129968,90,Domaine Gresser 2013 Kritt Gewurztraminer (Als...,Well-drained gravel soil gives this wine its c...,Roger Voss,@vossroger,30.0,Kritt,Gewürztraminer,Alsace,None,Alsace,France,Domaine Gresser
129969,90,Domaine Marcel Deiss 2012 Pinot Gris (Alsace),"A dry style of Pinot Gris, this is crisp with ...",Roger Voss,@vossroger,32.0,None,Pinot Gris,Alsace,None,Alsace,France,Domaine Marcel Deiss


In [28]:
# umplere goluri
df['country'] = df['country'].fillna("Missing") #ffill, bfill

#filtrare - să vedem dacă au mai rămas goluri
df[ df['country'].isna() ]

,points,title,description,taster_name,taster_twitter_handle,price,designation,variety,region_1,region_2,province,country,winery


Procedăm similar cu coloana `variety`.

In [29]:
df['variety'].fillna("Missing", inplace = True)
df[df['variety'].isna()]

,points,title,description,taster_name,taster_twitter_handle,price,designation,variety,region_1,region_2,province,country,winery


## Operații cu linii

In [30]:
# head - începutul tabelei ; tail - coada/sfărșitul tabelei
df.head(10)

,points,title,description,taster_name,taster_twitter_handle,price,designation,variety,region_1,region_2,province,country,winery
0,87,Nicosia 2013 Vulkà Bianco (Etna),"Aromas include tropical fruit, broom, brimston...",Kerin O’Keefe,@kerinokeefe,NaN,Vulkà Bianco,White Blend,Etna,None,Sicily & Sardinia,Italy,Nicosia
1,87,Quinta dos Avidagos 2011 Avidagos Red (Douro),"This is ripe and fruity, a wine that is smooth...",Roger Voss,@vossroger,15.0,Avidagos,Portuguese Red,None,None,Douro,Portugal,Quinta dos Avidagos
2,87,Rainstorm 2013 Pinot Gris (Willamette Valley),"Tart and snappy, the flavors of lime flesh and...",Paul Gregutt,@paulgwine,14.0,None,Pinot Gris,Willamette Valley,Willamette Valley,Oregon,US,Rainstorm
3,87,St. Julian 2013 Reserve Late Harvest Riesling ...,"Pineapple rind, lemon pith and orange blossom ...",Alexander Peartree,None,13.0,Reserve Late Harvest,Riesling,Lake Michigan Shore,None,Michigan,US,St. Julian
4,87,Sweet Cheeks 2012 Vintner's Reserve Wild Child...,"Much like the regular bottling from 2012, this...",Paul Gregutt,@paulgwine,65.0,Vintner's Reserve Wild Child Block,Pinot Noir,Willamette Valley,Willamette Valley,Oregon,US,Sweet Cheeks
5,87,Tandem 2011 Ars In Vitro Tempranillo-Merlot (N...,Blackberry and raspberry aromas show a typical...,Michael Schachner,@wineschach,15.0,Ars In Vitro,Tempranillo-Merlot,Navarra,None,Northern Spain,Spain,Tandem
6,87,Terre di Giurfo 2013 Belsito Frappato (Vittoria),"Here's a bright, informal red that opens with ...",Kerin O’Keefe,@kerinokeefe,16.0,Belsito,Frappato,Vittoria,None,Sicily & Sardinia,Italy,Terre di Giurfo
7,87,Trimbach 2012 Gewurztraminer (Alsace),This dry and restrained wine offers spice in p...,Roger Voss,@vossroger,24.0,None,Gewürztraminer,Alsace,None,Alsace,France,Trimbach
8,87,Heinz Eifel 2013 Shine Gewürztraminer (Rheinhe...,Savory dried thyme notes accent sunnier flavor...,Anna Lee C. Iijima,None,12.0,Shine,Gewürztraminer,None,None,Rheinhessen,Germany,Heinz Eifel
9,87,Jean-Baptiste Adam 2012 Les Natures Pinot Gris...,This has great depth of flavor with its fresh ...,Roger Voss,@vossroger,27.0,Les Natures,Pinot Gris,Alsace,None,Alsace,France,Jean-Baptiste Adam


In [31]:
df.tail(5)

,points,title,description,taster_name,taster_twitter_handle,price,designation,variety,region_1,region_2,province,country,winery
129966,90,Dr. H. Thanisch (Erben Müller-Burggraef) 2013 ...,Notes of honeysuckle and cantaloupe sweeten th...,Anna Lee C. Iijima,None,28.0,Brauneberger Juffer-Sonnenuhr Spätlese,Riesling,None,None,Mosel,Germany,Dr. H. Thanisch (Erben Müller-Burggraef)
129967,90,Citation 2004 Pinot Noir (Oregon),Citation is given as much as a decade of bottl...,Paul Gregutt,@paulgwine,75.0,None,Pinot Noir,Oregon,Oregon Other,Oregon,US,Citation
129968,90,Domaine Gresser 2013 Kritt Gewurztraminer (Als...,Well-drained gravel soil gives this wine its c...,Roger Voss,@vossroger,30.0,Kritt,Gewürztraminer,Alsace,None,Alsace,France,Domaine Gresser
129969,90,Domaine Marcel Deiss 2012 Pinot Gris (Alsace),"A dry style of Pinot Gris, this is crisp with ...",Roger Voss,@vossroger,32.0,None,Pinot Gris,Alsace,None,Alsace,France,Domaine Marcel Deiss
129970,90,Domaine Schoffit 2012 Lieu-dit Harth Cuvée Car...,"Big, rich and off-dry, this is powered by inte...",Roger Voss,@vossroger,21.0,Lieu-dit Harth Cuvée Caroline,Gewürztraminer,Alsace,None,Alsace,France,Domaine Schoffit


In [32]:
df

,points,title,description,taster_name,taster_twitter_handle,price,designation,variety,region_1,region_2,province,country,winery
0,87,Nicosia 2013 Vulkà Bianco (Etna),"Aromas include tropical fruit, broom, brimston...",Kerin O’Keefe,@kerinokeefe,NaN,Vulkà Bianco,White Blend,Etna,None,Sicily & Sardinia,Italy,Nicosia
1,87,Quinta dos Avidagos 2011 Avidagos Red (Douro),"This is ripe and fruity, a wine that is smooth...",Roger Voss,@vossroger,15.0,Avidagos,Portuguese Red,None,None,Douro,Portugal,Quinta dos Avidagos
2,87,Rainstorm 2013 Pinot Gris (Willamette Valley),"Tart and snappy, the flavors of lime flesh and...",Paul Gregutt,@paulgwine,14.0,None,Pinot Gris,Willamette Valley,Willamette Valley,Oregon,US,Rainstorm
3,87,St. Julian 2013 Reserve Late Harvest Riesling ...,"Pineapple rind, lemon pith and orange blossom ...",Alexander Peartree,None,13.0,Reserve Late Harvest,Riesling,Lake Michigan Shore,None,Michigan,US,St. Julian
4,87,Sweet Cheeks 2012 Vintner's Reserve Wild Child...,"Much like the regular bottling from 2012, this...",Paul Gregutt,@paulgwine,65.0,Vintner's Reserve Wild Child Block,Pinot Noir,Willamette Valley,Willamette Valley,Oregon,US,Sweet Cheeks
...,...,...,...,...,...,...,...,...,...,...,...,...,...
129966,90,Dr. H. Thanisch (Erben Müller-Burggraef) 2013 ...,Notes of honeysuckle and cantaloupe sweeten th...,Anna Lee C. Iijima,None,28.0,Brauneberger Juffer-Sonnenuhr Spätlese,Riesling,None,None,Mosel,Germany,Dr. H. Thanisch (Erben Müller-Burggraef)
129967,90,Citation 2004 Pinot Noir (Oregon),Citation is given as much as a decade of bottl...,Paul Gregutt,@paulgwine,75.0,None,Pinot Noir,Oregon,Oregon Other,Oregon,US,Citation
129968,90,Domaine Gresser 2013 Kritt Gewurztraminer (Als...,Well-drained gravel soil gives this wine its c...,Roger Voss,@vossroger,30.0,Kritt,Gewürztraminer,Alsace,None,Alsace,France,Domaine Gresser
129969,90,Domaine Marcel Deiss 2012 Pinot Gris (Alsace),"A dry style of Pinot Gris, this is crisp with ...",Roger Voss,@vossroger,32.0,None,Pinot Gris,Alsace,None,Alsace,France,Domaine Marcel Deiss


`info point` Într-o celulă, folosind Python, fără necesitatea de a invoca o metodă specifică, Jupyter va încerca să afișeze rezultatul ultimului calcul. Mai sus, metodele <b>tail, head</b> returnează un dataframe. De aceea, fiind fiecare ultima instrucțiune din celulă, Jupyter a afișat obiectul așa cum a putut mai bine.

Observați că șirurile mari de caractere nu sunt prezentate integral și că a apărut un control în partea de jos a ferestrei cu rezultate pentru a facilita inspectarea completă a coloanelor care nu au încăput.

Revenind la cum se pot publica rezultate de calcul dintr-o celulă - pentru printarea obișnuită se folosește `print` care este destul de complexă, admite formatări, dar ca și impact vizual nu e fantastic. Pentru afișarea tabelelor puteți folosi `display`.

De exemplu, dacă vrem sa vedem primele două și ultimele două linii în aceeiași fereastră de rezultate:

In [33]:
#display(df.tail(2))
df.tail(2)

,points,title,description,taster_name,taster_twitter_handle,price,designation,variety,region_1,region_2,province,country,winery
129969,90,Domaine Marcel Deiss 2012 Pinot Gris (Alsace),"A dry style of Pinot Gris, this is crisp with ...",Roger Voss,@vossroger,32.0,None,Pinot Gris,Alsace,None,Alsace,France,Domaine Marcel Deiss
129970,90,Domaine Schoffit 2012 Lieu-dit Harth Cuvée Car...,"Big, rich and off-dry, this is powered by inte...",Roger Voss,@vossroger,21.0,Lieu-dit Harth Cuvée Caroline,Gewürztraminer,Alsace,None,Alsace,France,Domaine Schoffit


In [34]:
print('Primele două linii')
display(df.head(2))
print('Ultimele două linii')
df.tail(2)

Primele două linii


,points,title,description,taster_name,taster_twitter_handle,price,designation,variety,region_1,region_2,province,country,winery
0,87,Nicosia 2013 Vulkà Bianco (Etna),"Aromas include tropical fruit, broom, brimston...",Kerin O’Keefe,@kerinokeefe,NaN,Vulkà Bianco,White Blend,Etna,None,Sicily & Sardinia,Italy,Nicosia
1,87,Quinta dos Avidagos 2011 Avidagos Red (Douro),"This is ripe and fruity, a wine that is smooth...",Roger Voss,@vossroger,15.0,Avidagos,Portuguese Red,None,None,Douro,Portugal,Quinta dos Avidagos


Ultimele două linii


,points,title,description,taster_name,taster_twitter_handle,price,designation,variety,region_1,region_2,province,country,winery
129969,90,Domaine Marcel Deiss 2012 Pinot Gris (Alsace),"A dry style of Pinot Gris, this is crisp with ...",Roger Voss,@vossroger,32.0,None,Pinot Gris,Alsace,None,Alsace,France,Domaine Marcel Deiss
129970,90,Domaine Schoffit 2012 Lieu-dit Harth Cuvée Car...,"Big, rich and off-dry, this is powered by inte...",Roger Voss,@vossroger,21.0,Lieu-dit Harth Cuvée Caroline,Gewürztraminer,Alsace,None,Alsace,France,Domaine Schoffit


Pentru primele trei instrucțiuni am folosit explicit metoda de afișare. Pentru ultima instrucțiune am lăsat decizia la Jupyter. Iată trei moduri diferite de prezentare a unui dfȘ

Explicit, folosint `print`

In [35]:
print(df.head(2))

   points                                          title  \
0      87              Nicosia 2013 Vulkà Bianco  (Etna)   
1      87  Quinta dos Avidagos 2011 Avidagos Red (Douro)   

                                         description    taster_name  \
0  Aromas include tropical fruit, broom, brimston...  Kerin O’Keefe   
1  This is ripe and fruity, a wine that is smooth...     Roger Voss   

  taster_twitter_handle  price   designation         variety region_1  \
0          @kerinokeefe    NaN  Vulkà Bianco     White Blend     Etna   
1            @vossroger   15.0      Avidagos  Portuguese Red     None   

  region_2           province   country               winery  
0     None  Sicily & Sardinia     Italy              Nicosia  
1     None              Douro  Portugal  Quinta dos Avidagos  


Explicit, invocând `display`

In [36]:
display(df.head(2))

,points,title,description,taster_name,taster_twitter_handle,price,designation,variety,region_1,region_2,province,country,winery
0,87,Nicosia 2013 Vulkà Bianco (Etna),"Aromas include tropical fruit, broom, brimston...",Kerin O’Keefe,@kerinokeefe,NaN,Vulkà Bianco,White Blend,Etna,None,Sicily & Sardinia,Italy,Nicosia
1,87,Quinta dos Avidagos 2011 Avidagos Red (Douro),"This is ripe and fruity, a wine that is smooth...",Roger Voss,@vossroger,15.0,Avidagos,Portuguese Red,None,None,Douro,Portugal,Quinta dos Avidagos


Implicit, default, cum știe Jupyter mai bine:

In [37]:
df.head(2)

,points,title,description,taster_name,taster_twitter_handle,price,designation,variety,region_1,region_2,province,country,winery
0,87,Nicosia 2013 Vulkà Bianco (Etna),"Aromas include tropical fruit, broom, brimston...",Kerin O’Keefe,@kerinokeefe,NaN,Vulkà Bianco,White Blend,Etna,None,Sicily & Sardinia,Italy,Nicosia
1,87,Quinta dos Avidagos 2011 Avidagos Red (Douro),"This is ripe and fruity, a wine that is smooth...",Roger Voss,@vossroger,15.0,Avidagos,Portuguese Red,None,None,Douro,Portugal,Quinta dos Avidagos


Selectarea unei linii prin număr de referință.

In [38]:
# selectarea liniei cu indexul 7 - poveste mai lungă, cine e indexul etc. Aici știm că indexul e numeric și că începe de la 0

print('selectarea direct folosind numărul înregistrării', df.iloc[7])

selectarea direct folosind numărul înregistrării points                                                                  87
title                                Trimbach 2012 Gewurztraminer (Alsace)
description              This dry and restrained wine offers spice in p...
taster_name                                                     Roger Voss
taster_twitter_handle                                           @vossroger
price                                                                 24.0
designation                                                           None
variety                                                     Gewürztraminer
region_1                                                            Alsace
region_2                                                              None
province                                                            Alsace
country                                                             France
winery                                             

Alternativ, dacă punem condiția ca indexul să fie egal cu 7

In [39]:
df[df.index == 7]

,points,title,description,taster_name,taster_twitter_handle,price,designation,variety,region_1,region_2,province,country,winery
7,87,Trimbach 2012 Gewurztraminer (Alsace),This dry and restrained wine offers spice in p...,Roger Voss,@vossroger,24.0,None,Gewürztraminer,Alsace,None,Alsace,France,Trimbach


Alte alternative de selectare?



## Filtrare/Selectare condiționată

In [40]:
# Folosirea parantezelor pătrate indică selecție
df[df.taster_name == 'Roger Voss']

,points,title,description,taster_name,taster_twitter_handle,price,designation,variety,region_1,region_2,province,country,winery
1,87,Quinta dos Avidagos 2011 Avidagos Red (Douro),"This is ripe and fruity, a wine that is smooth...",Roger Voss,@vossroger,15.0,Avidagos,Portuguese Red,None,None,Douro,Portugal,Quinta dos Avidagos
7,87,Trimbach 2012 Gewurztraminer (Alsace),This dry and restrained wine offers spice in p...,Roger Voss,@vossroger,24.0,None,Gewürztraminer,Alsace,None,Alsace,France,Trimbach
9,87,Jean-Baptiste Adam 2012 Les Natures Pinot Gris...,This has great depth of flavor with its fresh ...,Roger Voss,@vossroger,27.0,Les Natures,Pinot Gris,Alsace,None,Alsace,France,Jean-Baptiste Adam
11,87,Leon Beyer 2012 Gewurztraminer (Alsace),"This is a dry wine, very spicy, with a tight, ...",Roger Voss,@vossroger,30.0,None,Gewürztraminer,Alsace,None,Alsace,France,Leon Beyer
30,86,Domaine de la Madone 2012 Nouveau (Beaujolais...,Red cherry fruit comes laced with light tannin...,Roger Voss,@vossroger,NaN,Nouveau,Gamay,Beaujolais-Villages,None,Beaujolais,France,Domaine de la Madone
...,...,...,...,...,...,...,...,...,...,...,...,...,...
129964,90,Domaine Ehrhart 2013 Domaine Saint-Rémy Herren...,"Initially quite muted, this wine slowly develo...",Roger Voss,@vossroger,NaN,Domaine Saint-Rémy Herrenweg,Gewürztraminer,Alsace,None,Alsace,France,Domaine Ehrhart
129965,90,Domaine Rieflé-Landmann 2013 Seppi Landmann Va...,"While it's rich, this beautiful dry wine also ...",Roger Voss,@vossroger,28.0,Seppi Landmann Vallée Noble,Pinot Gris,Alsace,None,Alsace,France,Domaine Rieflé-Landmann
129968,90,Domaine Gresser 2013 Kritt Gewurztraminer (Als...,Well-drained gravel soil gives this wine its c...,Roger Voss,@vossroger,30.0,Kritt,Gewürztraminer,Alsace,None,Alsace,France,Domaine Gresser
129969,90,Domaine Marcel Deiss 2012 Pinot Gris (Alsace),"A dry style of Pinot Gris, this is crisp with ...",Roger Voss,@vossroger,32.0,None,Pinot Gris,Alsace,None,Alsace,France,Domaine Marcel Deiss


### Conditii compuse - specificare OR, AND, paranteze rotunde

Când vrema să folosim o condiție mai complexă de filtrare vom folosi paranteze rotunde pentru a delimita condițiile simple pe care le folosim și operatorii <b>SAU</b> = `|` și <b>ȘI</b> = `&`.

De exemplu dacă vrem să vedem notele peste 80 sau sub 20 date de Roger Voss:



In [41]:
df[  ( 
        (df['price'] > 80) | (df['price' ] < 20 ) # or = |
     ) &  # and = &
    (df.taster_name == 'Roger Voss') ] # accesare coloana cu punct

,points,title,description,taster_name,taster_twitter_handle,price,designation,variety,region_1,region_2,province,country,winery
1,87,Quinta dos Avidagos 2011 Avidagos Red (Douro),"This is ripe and fruity, a wine that is smooth...",Roger Voss,@vossroger,15.0,Avidagos,Portuguese Red,None,None,Douro,Portugal,Quinta dos Avidagos
42,86,Henry Fessy 2012 Nouveau (Beaujolais),"This is a festive wine, with soft, ripe fruit ...",Roger Voss,@vossroger,9.0,Nouveau,Gamay,Beaujolais,None,Beaujolais,France,Henry Fessy
49,86,Vignerons de Bel Air 2011 Eté Indien (Brouilly),"Soft and fruity, this is a generous, ripe wine...",Roger Voss,@vossroger,14.0,Eté Indien,Gamay,Brouilly,None,Beaujolais,France,Vignerons de Bel Air
53,85,Château de Sours 2011 La Fleur d'Amélie (Bord...,"Fruity and lightly herbaceous, this has fine t...",Roger Voss,@vossroger,15.0,La Fleur d'Amélie,Bordeaux-style White Blend,Bordeaux Blanc,None,Bordeaux,France,Château de Sours
66,86,Vignerons des Terres Secrètes 2015 Mâcon-Mill...,"This soft, rounded wine is ripe with generous ...",Roger Voss,@vossroger,15.0,None,Chardonnay,Mâcon-Milly Lamartine,None,Burgundy,France,Vignerons des Terres Secrètes
...,...,...,...,...,...,...,...,...,...,...,...,...,...
129845,86,Höpler 2007 Grüner Veltliner (Burgenland),"Höpler's entry-level Grüner is fresh, green, s...",Roger Voss,@vossroger,13.0,None,Grüner Veltliner,None,None,Burgenland,Austria,Höpler
129865,90,Sogrape 2014 Silk & Spice Red (Portuguese Tabl...,Celebrating the Portuguese discovery of the sp...,Roger Voss,@vossroger,15.0,Silk & Spice,Portuguese Red,None,None,Portuguese Table Wine,Portugal,Sogrape
129876,90,Domaine des Terres Dorées 2015 L'Ancien (Beau...,"This is a delicious, ripe and fruity wine that...",Roger Voss,@vossroger,19.0,L'Ancien,Gamay,Beaujolais,None,Beaujolais,France,Domaine des Terres Dorées
129894,91,Lionel Osmin & Cie 2014 Abouriou (Côtes du Mar...,"Despite its proximity to Bordeaux, the Marmand...",Roger Voss,@vossroger,15.0,None,Abouriou,Côtes du Marmandais,None,Southwest France,France,Lionel Osmin & Cie


`info point` Am folosit două moduri diferite de accesare a coloanelor: după nume și ca și proprietate a obiectului <b>df</b>.

## Operații cu coloane

Selectarea unei singure coloane se poate face exact cum am menționat mai sus, în cel puțin două moduri.


Selectarea mai multor coloane

In [42]:
# selectare cu index

print('Toate liniile, coloana a 2-a (se începe de la 0...)\n', df.iloc[:,2])

# selectare cu liste de indecsi
print('\n','Primele 5 linii, coloanele 0,2,4\n', df.iloc[0:5,[0,2,4]])
# selectare cu :

print('\n','Primele 5 linii, coloanele 2,3,4\n', df.iloc[0:5,2:5])

Toate liniile, coloana a 2-a (se începe de la 0...)
 0         Aromas include tropical fruit, broom, brimston...
1         This is ripe and fruity, a wine that is smooth...
2         Tart and snappy, the flavors of lime flesh and...
3         Pineapple rind, lemon pith and orange blossom ...
4         Much like the regular bottling from 2012, this...
                                ...                        
129966    Notes of honeysuckle and cantaloupe sweeten th...
129967    Citation is given as much as a decade of bottl...
129968    Well-drained gravel soil gives this wine its c...
129969    A dry style of Pinot Gris, this is crisp with ...
129970    Big, rich and off-dry, this is powered by inte...
Name: description, Length: 129971, dtype: object

 Primele 5 linii, coloanele 0,2,4
    points                                        description  \
0      87  Aromas include tropical fruit, broom, brimston...   
1      87  This is ripe and fruity, a wine that is smooth...   
2      87 

### Selectarea coloanelor folosind liste cu nume de coloane

- folositor de exemplu la group by sau la pivotare

In [43]:
df.columns

Index(['points', 'title', 'description', 'taster_name',
       'taster_twitter_handle', 'price', 'designation', 'variety', 'region_1',
       'region_2', 'province', 'country', 'winery'],
      dtype='object')

In [44]:
coloane_de_interes = ['country','description','price']

df[ coloane_de_interes ]


,country,description,price
0,Italy,"Aromas include tropical fruit, broom, brimston...",NaN
1,Portugal,"This is ripe and fruity, a wine that is smooth...",15.0
2,US,"Tart and snappy, the flavors of lime flesh and...",14.0
3,US,"Pineapple rind, lemon pith and orange blossom ...",13.0
4,US,"Much like the regular bottling from 2012, this...",65.0
...,...,...,...
129966,Germany,Notes of honeysuckle and cantaloupe sweeten th...,28.0
129967,US,Citation is given as much as a decade of bottl...,75.0
129968,France,Well-drained gravel soil gives this wine its c...,30.0
129969,France,"A dry style of Pinot Gris, this is crisp with ...",32.0


## Redenumirea coloanelor

In [45]:
df['nume_de_schimbat'] = 0
df

,points,title,description,taster_name,taster_twitter_handle,price,designation,variety,region_1,region_2,province,country,winery,nume_de_schimbat
0,87,Nicosia 2013 Vulkà Bianco (Etna),"Aromas include tropical fruit, broom, brimston...",Kerin O’Keefe,@kerinokeefe,NaN,Vulkà Bianco,White Blend,Etna,None,Sicily & Sardinia,Italy,Nicosia,0
1,87,Quinta dos Avidagos 2011 Avidagos Red (Douro),"This is ripe and fruity, a wine that is smooth...",Roger Voss,@vossroger,15.0,Avidagos,Portuguese Red,None,None,Douro,Portugal,Quinta dos Avidagos,0
2,87,Rainstorm 2013 Pinot Gris (Willamette Valley),"Tart and snappy, the flavors of lime flesh and...",Paul Gregutt,@paulgwine,14.0,None,Pinot Gris,Willamette Valley,Willamette Valley,Oregon,US,Rainstorm,0
3,87,St. Julian 2013 Reserve Late Harvest Riesling ...,"Pineapple rind, lemon pith and orange blossom ...",Alexander Peartree,None,13.0,Reserve Late Harvest,Riesling,Lake Michigan Shore,None,Michigan,US,St. Julian,0
4,87,Sweet Cheeks 2012 Vintner's Reserve Wild Child...,"Much like the regular bottling from 2012, this...",Paul Gregutt,@paulgwine,65.0,Vintner's Reserve Wild Child Block,Pinot Noir,Willamette Valley,Willamette Valley,Oregon,US,Sweet Cheeks,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
129966,90,Dr. H. Thanisch (Erben Müller-Burggraef) 2013 ...,Notes of honeysuckle and cantaloupe sweeten th...,Anna Lee C. Iijima,None,28.0,Brauneberger Juffer-Sonnenuhr Spätlese,Riesling,None,None,Mosel,Germany,Dr. H. Thanisch (Erben Müller-Burggraef),0
129967,90,Citation 2004 Pinot Noir (Oregon),Citation is given as much as a decade of bottl...,Paul Gregutt,@paulgwine,75.0,None,Pinot Noir,Oregon,Oregon Other,Oregon,US,Citation,0
129968,90,Domaine Gresser 2013 Kritt Gewurztraminer (Als...,Well-drained gravel soil gives this wine its c...,Roger Voss,@vossroger,30.0,Kritt,Gewürztraminer,Alsace,None,Alsace,France,Domaine Gresser,0
129969,90,Domaine Marcel Deiss 2012 Pinot Gris (Alsace),"A dry style of Pinot Gris, this is crisp with ...",Roger Voss,@vossroger,32.0,None,Pinot Gris,Alsace,None,Alsace,France,Domaine Marcel Deiss,0


Pentru redenumire vom folosi metoda `rename` cu parametrul `columns` egal cu un dicționar cu chei numele vechi și valori numele noilor coloane. Opțional se poate folosi `inplace = True` pentru ca schimbarea să se regăseasca în dataframe; în caz contrar operația de redenumire va da naștere unui nou dataframe.

In [46]:
df.rename(columns = {'nume_de_schimbat': 'nume_nou'},inplace = True) 
# ALTERNATIV: df = df.rename(columns = {'nume_de_schimbat': 'nume_nou'})
df

,points,title,description,taster_name,taster_twitter_handle,price,designation,variety,region_1,region_2,province,country,winery,nume_nou
0,87,Nicosia 2013 Vulkà Bianco (Etna),"Aromas include tropical fruit, broom, brimston...",Kerin O’Keefe,@kerinokeefe,NaN,Vulkà Bianco,White Blend,Etna,None,Sicily & Sardinia,Italy,Nicosia,0
1,87,Quinta dos Avidagos 2011 Avidagos Red (Douro),"This is ripe and fruity, a wine that is smooth...",Roger Voss,@vossroger,15.0,Avidagos,Portuguese Red,None,None,Douro,Portugal,Quinta dos Avidagos,0
2,87,Rainstorm 2013 Pinot Gris (Willamette Valley),"Tart and snappy, the flavors of lime flesh and...",Paul Gregutt,@paulgwine,14.0,None,Pinot Gris,Willamette Valley,Willamette Valley,Oregon,US,Rainstorm,0
3,87,St. Julian 2013 Reserve Late Harvest Riesling ...,"Pineapple rind, lemon pith and orange blossom ...",Alexander Peartree,None,13.0,Reserve Late Harvest,Riesling,Lake Michigan Shore,None,Michigan,US,St. Julian,0
4,87,Sweet Cheeks 2012 Vintner's Reserve Wild Child...,"Much like the regular bottling from 2012, this...",Paul Gregutt,@paulgwine,65.0,Vintner's Reserve Wild Child Block,Pinot Noir,Willamette Valley,Willamette Valley,Oregon,US,Sweet Cheeks,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
129966,90,Dr. H. Thanisch (Erben Müller-Burggraef) 2013 ...,Notes of honeysuckle and cantaloupe sweeten th...,Anna Lee C. Iijima,None,28.0,Brauneberger Juffer-Sonnenuhr Spätlese,Riesling,None,None,Mosel,Germany,Dr. H. Thanisch (Erben Müller-Burggraef),0
129967,90,Citation 2004 Pinot Noir (Oregon),Citation is given as much as a decade of bottl...,Paul Gregutt,@paulgwine,75.0,None,Pinot Noir,Oregon,Oregon Other,Oregon,US,Citation,0
129968,90,Domaine Gresser 2013 Kritt Gewurztraminer (Als...,Well-drained gravel soil gives this wine its c...,Roger Voss,@vossroger,30.0,Kritt,Gewürztraminer,Alsace,None,Alsace,France,Domaine Gresser,0
129969,90,Domaine Marcel Deiss 2012 Pinot Gris (Alsace),"A dry style of Pinot Gris, this is crisp with ...",Roger Voss,@vossroger,32.0,None,Pinot Gris,Alsace,None,Alsace,France,Domaine Marcel Deiss,0


Ștergerea unei coloane sau a unei liste de coloane - observați din nou folosire lui `inplace = True`.

In [47]:

# verificăm dacă mai e în df coloana pe care vrem să o ștergem:
if 'nume_nou' in df.columns:
    
    # apelam metoda drop care poate șterge o listă de coloane specificată în parametrul columns
    df.drop(columns = ['nume_nou'], inplace = True)
    
df.columns

Index(['points', 'title', 'description', 'taster_name',
       'taster_twitter_handle', 'price', 'designation', 'variety', 'region_1',
       'region_2', 'province', 'country', 'winery'],
      dtype='object')

## Sortarea unui dataframe

Se face cu metoda pandas `sort_value` specificând parametrii:
- <b>by</b>=listă nume coloane după care să se facă sortarea, în ordinea dorită
- <b>ascending</b>=listă de valori True/False corespunzătoare coloanelor dorite în sortare. O valoare True pe poziția i în această listă înseamnă sortare crescătoare pentru coloana i din lista <b>by</b>.

Implicit sortarea e crescătoare. Primul exemplu specifică doar o coloană, caz în care nici nu e nevoie de listă (!):

In [48]:
df = df.sort_values ( by = 'country') # țările alfabetic, crecător

In [49]:
df = df.sort_values ( by = 'country',
                    ascending =False ) # țările alfabetic, descrecător, ascending are valoarea implictă = True

In [50]:
df = df.sort_values ( by = ['country','points'],
                    ascending = [False, False]
                    ) # țările alfabetic, vinurile cele mai bune la început
df

,points,title,description,taster_name,taster_twitter_handle,price,designation,variety,region_1,region_2,province,country,winery
18460,92,Bouza 2015 B6 Parcela Única Tannat (Canelones),Dried blackberry aromas include welcome notes ...,Michael Schachner,@wineschach,43.0,B6 Parcela Única,Tannat,None,None,Canelones,Uruguay,Bouza
14119,92,Bouza 2015 B6 Parcela Única Tannat (Canelones),Dried blackberry aromas include welcome notes ...,Michael Schachner,@wineschach,43.0,B6 Parcela Única,Tannat,None,None,Canelones,Uruguay,Bouza
98839,91,Pizzorno 2011 Primo Red (Uruguay),"Spicy oak aromas suggest sandalwood, while thi...",Michael Schachner,@wineschach,50.0,Primo,Red Blend,None,None,Uruguay,Uruguay,Pizzorno
39361,91,Narbona NV Blend 002 Limited Edition Tannat-Ca...,"Cherry and berry aromas are ripe, healthy and ...",Michael Schachner,@wineschach,22.0,Blend 002 Limited Edition,Tannat-Cabernet Franc,None,None,Uruguay,Uruguay,Narbona
39373,91,Bouza 2015 Reserva Tannat (Montevideo),If you're going to try just one Uruguayan Tann...,Michael Schachner,@wineschach,22.0,Reserva,Tannat,None,None,Montevideo,Uruguay,Bouza
...,...,...,...,...,...,...,...,...,...,...,...,...,...
128262,80,Reyter 2009 Finca La Martina Chardonnay (Mendoza),"Smells like crushed vitamins, iron and mineral...",Michael Schachner,@wineschach,13.0,Finca La Martina,Chardonnay,Mendoza,None,Mendoza Province,Argentina,Reyter
128255,80,Viniterra 2007 Malbec (Mendoza),"Severely compromised by green, minty, weedy ar...",Michael Schachner,@wineschach,13.0,None,Malbec,Mendoza,None,Mendoza Province,Argentina,Viniterra
128254,80,François Lurton 2006 Gran Lurton Cabernet Sauv...,Disappointing considering the source. The nose...,Michael Schachner,@wineschach,20.0,Gran Lurton,Cabernet Sauvignon,Mendoza,None,Mendoza Province,Argentina,François Lurton
111751,80,Bota Box 2011 Moscato (Mendoza),"This smells cheap, weird and almost dirty. The...",Michael Schachner,@wineschach,20.0,None,Moscato,Mendoza,None,Mendoza Province,Argentina,Bota Box


## Adăugarea unor linii noi

Vezi concat, prima parte, cu axis = 0.

## Editarea unor valori

In [51]:
df.iloc[0]

points                                                                  92
title                       Bouza 2015 B6 Parcela Única Tannat (Canelones)
description              Dried blackberry aromas include welcome notes ...
taster_name                                              Michael Schachner
taster_twitter_handle                                          @wineschach
price                                                                 43.0
designation                                               B6 Parcela Única
variety                                                             Tannat
region_1                                                              None
region_2                                                              None
province                                                         Canelones
country                                                            Uruguay
winery                                                               Bouza
Name: 18460, dtype: objec

Modific prețul și apoi modific la loc.

`info point` Modificările se fac pentru toate înregistrările care satisfac condiția impusă.

## .loc

In [52]:
df.loc[df.title.str.startswith('Bodega Catena Zapata 2006 Nicasia Vineyard Mal')]

,points,title,description,taster_name,taster_twitter_handle,price,designation,variety,region_1,region_2,province,country,winery
82754,97,Bodega Catena Zapata 2006 Nicasia Vineyard Mal...,"If the color doesn't tell the full story, the ...",Michael Schachner,@wineschach,120.0,Nicasia Vineyard,Malbec,Mendoza,None,Mendoza Province,Argentina,Bodega Catena Zapata


In [53]:
df.loc[ df.title.str.startswith('Bodega Catena Zapata 2006 Nicasia Vineyard Mal'),'price' ] = 0
df.loc[df.title.str.startswith('Bodega Catena Zapata 2006 Nicasia Vineyard Mal')]

,points,title,description,taster_name,taster_twitter_handle,price,designation,variety,region_1,region_2,province,country,winery
82754,97,Bodega Catena Zapata 2006 Nicasia Vineyard Mal...,"If the color doesn't tell the full story, the ...",Michael Schachner,@wineschach,0.0,Nicasia Vineyard,Malbec,Mendoza,None,Mendoza Province,Argentina,Bodega Catena Zapata


In [54]:
df.loc[df.title.str.startswith('Bodega Catena Zapata 2006 Nicasia Vineyard Mal'),'price'] = 120
df.loc[df.title.str.startswith('Bodega Catena Zapata 2006 Nicasia Vineyard Mal')]

,points,title,description,taster_name,taster_twitter_handle,price,designation,variety,region_1,region_2,province,country,winery
82754,97,Bodega Catena Zapata 2006 Nicasia Vineyard Mal...,"If the color doesn't tell the full story, the ...",Michael Schachner,@wineschach,120.0,Nicasia Vineyard,Malbec,Mendoza,None,Mendoza Province,Argentina,Bodega Catena Zapata


### Exemplu: 

După cum vedeți s-a strecurat un typo - `Uncofirmed` care a dus la apariția unei categorii noi, în mod eronat. Editați eroarea și coretați-o cu `Unconfirmed` selectând coloana și condiționând ca valoarea să fie cea incorectă (`Uncofirmed`), și atribuind acestor înregistrări valoarea  corectă `Unconfirmed`.

In [55]:
file_name = './date/monitor_one_daily_output.csv'

# citire fisierului - 'df' de la dataframe, si un nume de variabila:
df_sample_csv = pd.read_csv ( file_name)

# verificam primele linii invocand functia head:
df_sample_csv.Belief.drop_duplicates()

În acest exemplu voi folosi un șablon boolean, de condiționare a liniilor.
După virgulă se pot selecta coloanele dorite.

Când se selectează o singură coloană, nu e nevoie de lista.

Prima dată putem vedea numărul liniilor care au problema:

In [ ]:
df_sample_csv.loc[ df_sample_csv['Belief'] == 'Uncofirmed'].shape

(49, 10)

Aplicăm corecția în acele linii:

In [ ]:
df_sample_csv.loc[ df_sample_csv['Belief'] == 'Uncofirmed','Belief'] = 'Unconfirmed'

Verificăm corectitudinea operației - vedem numărul liniilor care au problema:

In [ ]:
df_sample_csv[ df_sample_csv['Belief'] == 'Uncofirmed'].shape

(0, 10)

Alte exemple de selectare cu .loc

In [ ]:
df_sample_csv.loc[0:10,['Likelihood_Score','Belief']]

,Likelihood_Score,Belief
0,92.703030,Weak
1,79.393106,Unconfirmed
2,90.546950,Unconfirmed
3,99.561564,Unconfirmed
4,100.000000,Unconfirmed
5,99.733736,Unconfirmed
6,99.911712,Unconfirmed
7,74.836137,Unconfirmed
8,99.941313,Unconfirmed
9,99.790908,Unconfirmed


## .iloc

Selectare pe baza numărului de linie și de coloană. 

Se pot folosi intervale de linii si coloane. 

Prima dată se precizează liniile, urmate de virgulă și precizarea coloanelor.

### Exemple:

Prima linie, a doua coloană (indexul pozitional porneste de la 0!)

In [ ]:
df_sample_csv.iloc[0,1]

'2/24/2019'

O lista de linii si alta de coloane

In [ ]:
df_sample_csv.iloc[[0,3,7,8],[1,3,5]]

,date_only,Daily_Outcome_Idx,Over Heat Event
0,2/24/2019,2,NaN
3,2/20/2019,2,NaN
7,2/11/2019,0,NaN
8,2/6/2019,0,NaN


Atât pentru linii cât și pentru coloane se pot preciza intervale de valori `<început>:<sfârșit>` după cum urmează:

`DE REȚINUT`: în Python ultimul index din interval este omis - astfel, pentru intervalul de mai sus valorile considerate sunt `<început>`, `<început>+1`, ..., `<sfârșit>`-1

EXEMPLU:
- selectam a opta inregistrare si coloanele 2,3,4 - pentru asta intervalul indicilor de coloane este [2:<b>5</b>]:

In [ ]:
# înainte să vedem și numele coloanelor. Pentru asta folosim instrucțiune explicită de print,
# altfel nu se va vizualiza decât rezultatul ultimei linii de mai jos... 
print(df_sample_csv.columns[2:5])
df_sample_csv.iloc[7,2:5]

Index(['Daily_Outcome', 'Daily_Outcome_Idx', 'Likelihood_Score'], dtype='object')


Daily_Outcome        Overheat in less than 7 days
Daily_Outcome_Idx                               0
Likelihood_Score                        74.836137
Name: 7, dtype: object

- primele opt inregistrari si coloanele 2,3,4

In [ ]:
df_sample_csv.iloc[0:7,2:5]

,Daily_Outcome,Daily_Outcome_Idx,Likelihood_Score
0,Overheat in more than 14 days,2,92.703030
1,"Overheat in [7, 14] days",1,79.393106
2,Overheat in less than 7 days,0,90.546950
3,Overheat in more than 14 days,2,99.561564
4,Overheat in more than 14 days,2,100.000000
5,"Overheat in [7, 14] days",1,99.733736
6,Overheat in less than 7 days,0,99.911712


- ceva mai exotic: toate coloanele impare cu ajutorul unei expresii `lambda`

In [ ]:
df_sample_csv.iloc[ lambda x: x.index % 2 == 0, [0,1,3] ]

,car,date_only,Daily_Outcome_Idx
0,2221138,2/24/2019,2
2,2221138,2/22/2019,0
4,2221138,2/19/2019,2
6,2221138,2/12/2019,0
8,2221138,2/6/2019,0
...,...,...,...
142,2221251,12/13/2018,0
144,2221251,12/11/2018,0
146,2221251,12/9/2018,0
148,2221251,12/7/2018,0


# Lucrul cu mai multe dataframe-uri in pandas

## Metode de combinare:

Concatenare (pd.concat - axis = 0 sau axis = 1)

Join/Merge
- intern (chei comune)
- extern

    -extern pe stanga (toate cheile de pe stanga)
    
    -extern pe dreapta (toate cheile de pe dreapta)

# Concatenarea unei liste de dataframe-uri

Se poate face indiferent de structura, insa mare atentie:

- daca toate dataframe-urile au acelasi header, adica aceleasi coloane, rezultatul este reuniunea seturilor de date

- daca structura este diferita, atunci la final dataframe-ul rezultat are ca si coloane reuniunea tuturor coloanelor ce apartin dataframe-urilor din lista; acolo unde pentru un dataframe din lista lipsea o coloana din cele de la final, se va umple cu None (valoare lipsa)

NOTA:
Nu intotdeauna operatia de mai sus este dorita. 

# Exemplul 1 - concat

Am spus că vom folosi des dicționare - este un mod eficace de a trece de la o structură de listă la dataframe și înapoi. Aici vom crea dataframe-uri pornind de la o lista de dicționare cu aceleași chei (care vor deveni numele coloanelor).

In [ ]:
# constructie de data frame din lista de dictionare cu aceeasi structura

lst1 = [{'c1':i,'c2':10*i,'c3':2*i+1} for i in range(5)]
lst2 = [{'c1':i,'c3':10*i,'c2':2*(i-10)+1} for i in range(10,15)]
lst3 = [{'c1':i,'c2':100*i} for i in range(5)]
lst4 = [{'c1':i,'c4':7*i} for i in range(5)]

df1 = pd.DataFrame(lst1)
df2 = pd.DataFrame(lst2)
df3 = pd.DataFrame(lst3)
df4 = pd.DataFrame(lst4)

df1, df2, df3, df4

(   c1  c2  c3
 0   0   0   1
 1   1  10   3
 2   2  20   5
 3   3  30   7
 4   4  40   9,
    c1   c3  c2
 0  10  100   1
 1  11  110   3
 2  12  120   5
 3  13  130   7
 4  14  140   9,
    c1   c2
 0   0    0
 1   1  100
 2   2  200
 3   3  300
 4   4  400,
    c1  c4
 0   0   0
 1   1   7
 2   2  14
 3   3  21
 4   4  28)

Primul caz: aceleși nume de coloane, scrise doar în ordine diferită

In [ ]:
df_12 = pd.concat([df1,df2],sort = False)
display(df_12)

#....

,c1,c2,c3
0,0,0,1
1,1,10,3
2,2,20,5
3,3,30,7
4,4,40,9
0,10,1,100
1,11,3,110
2,12,5,120
3,13,7,130
4,14,9,140


Aici la df3 lipsește coloana c3:

In [ ]:
df_23 = pd.concat([df2,df3],sort = False)
display(df_23)

,c1,c3,c2
0,10,100.0,1
1,11,110.0,3
2,12,120.0,5
3,13,130.0,7
4,14,140.0,9
0,0,NaN,0
1,1,NaN,100
2,2,NaN,200
3,3,NaN,300
4,4,NaN,400


Încercăm un alt mod de umplere a valorilor lipsă, folosind funcții direct în linia de cod - lambda functions:

In [ ]:
df_23['c3'] = df_23['c3'].apply(lambda x: -1 if np.isnan(x) else x )

#df_23['c3'].fillna(-1,inplace= True)
df_23

,c1,c3,c2
0,10,100.0,1
1,11,110.0,3
2,12,120.0,5
3,13,130.0,7
4,14,140.0,9
0,0,-1.0,0
1,1,-1.0,100
2,2,-1.0,200
3,3,-1.0,300
4,4,-1.0,400


Un alt mod uzual de a imputa valori lipsă este folosind metoda `fillna`. Există și corespondent la operația de ștergere - `dropna`.

`info point` Există diferite tipuri de date `lipsă` - avem np.nan, pd.nan, None. Vă invit să vedeți singur cum/ când să le folosiți.

Ultimul caz de concatenare de acest tip, unul peste altul: doar prima coloana e comună. Și în acest caz se creeză toate coloanele posibile iar unde lipsesc valori se  introduce valoarea speicifică NaN.

In [ ]:
df_34=pd.concat([df3,df4],sort = False)
df_34

,c1,c2,c4
0,0,0.0,NaN
1,1,100.0,NaN
2,2,200.0,NaN
3,3,300.0,NaN
4,4,400.0,NaN
0,0,NaN,0.0
1,1,NaN,7.0
2,2,NaN,14.0
3,3,NaN,21.0
4,4,NaN,28.0


## Parametrul <i>axis</i>

Ne putem imagina două dataframe-uri cu același număr de înregistrări și pe care vrem să le alaturăm (nu să le suprapunem, ca mai sus).

Metoda concat are un parametru, axis, prezent și în alte contexte în care direcția acțiunii trebuie precizată. Implicit are valoarea 0 sau <i>lines</i> care indică faptul că vor avea loc operații pe linii. Cealalta opțiune este 1 sau <i>columns</i>.

În particular, dacă vrem să alăturam două dataframe-uri în acest fels iată cum procedăm:

In [ ]:
df5 = pd.DataFrame([{'c1':i,'c2':100*i} for i in range(5)])
df6 = pd.DataFrame([{'c3':i**2,'c4':7*i} for i in range(5)])

df56 = pd.concat([df5,df6],axis = 1)
df56

,c1,c2,c3,c4
0,0,0,0,0
1,1,100,1,7
2,2,200,4,14
3,3,300,9,21
4,4,400,16,28


Să vedem ce se întâmplă dacă nu avem același număr de linii:
    

In [ ]:
df7 = pd.DataFrame([{'c1':i,'c2':100*i} for i in range(5)])
df8 = pd.DataFrame([{'c3':i**2,'c4':7*i} for i in range(7)])

df78 = pd.concat([df7,df8],axis = 1)
df78

,c1,c2,c3,c4
0,0.0,0.0,0,0
1,1.0,100.0,1,7
2,2.0,200.0,4,14
3,3.0,300.0,9,21
4,4.0,400.0,16,28
5,NaN,NaN,25,35
6,NaN,NaN,36,42


## Merge - echivalentul lui "JOIN" din SQL 

Parametrii uzuali:

- `on`: lista  cu numele coloanelor in cazul in care au aceleasi nume in ambele dataframerui
- `left_on`, `right_on`: numele cheilor pe care se face operatia cand nu sunt la fel
- `how`: tipul operatiei - inner, right, left sau outer sunt valori posibile, similare cu operatiile din SQL

In [ ]:
df34_merge = df3.merge(df4, on = ['c1'], how = 'inner')
display(df34_merge)

,c1,c2,c4
0,0,0,0
1,1,100,7
2,2,200,14
3,3,300,21
4,4,400,28


In [ ]:
# merge df1 & df2 lefton, righton - change df2 to have results

## Introducere în lucrul cu groupby, transform și pivot

Desigur că mai tot ce se poate face în SQL la nivel de grupuri se poate obține și aici. 

#### .groupby

Groupby este un obiect foarte special cu multe moduri de lucru. Iată câteva exemple:

In [ ]:
df_tmp = df_34.fillna(0)

# grupăm după c1 și sumăm:
display(df_tmp.groupby('c1').sum())
display(df_tmp.groupby('c1', as_index = False).sum())
display(df_tmp.groupby('c1').sum().reset_index(drop=False))
display(df_tmp.groupby('c1').sum().reset_index(drop=True))

,c2,c4
c1,,
0,0.0,0.0
1,100.0,7.0
2,200.0,14.0
3,300.0,21.0
4,400.0,28.0


,c1,c2,c4
0,0,0.0,0.0
1,1,100.0,7.0
2,2,200.0,14.0
3,3,300.0,21.0
4,4,400.0,28.0


,c1,c2,c4
0,0,0.0,0.0
1,1,100.0,7.0
2,2,200.0,14.0
3,3,300.0,21.0
4,4,400.0,28.0


,c2,c4
0,0.0,0.0
1,100.0,7.0
2,200.0,14.0
3,300.0,21.0
4,400.0,28.0


După cum vedeți coloana c1 a devenit index. Puteți scăpa de această problemă fie apelând reset-index() la sfârșit, fie indicând la groupby as_index = False.

Sunt multe situațiile în care dorim să efectuăm diferite operații la nivel de grup, uneori aceleași cu toate coloanele, alteori diverse operații cu diferite coloane:

In [ ]:
df_tmp.groupby(['c1','c2'], as_index = False).max()

,c1,c2,c4
0,0,0.0,0.0
1,1,0.0,7.0
2,1,100.0,0.0
3,2,0.0,14.0
4,2,200.0,0.0
5,3,0.0,21.0
6,3,300.0,0.0
7,4,0.0,28.0
8,4,400.0,0.0


Putem grupa operații multiple asupra unei coloane - aceasta va duce la coloane multiindex, cu numele în tupluri. Putem aplica funcții utilizator la nivel de grup.

In [ ]:
df_tmp_agg = df_tmp.groupby(['c1'], as_index = False).agg(

    {
        'c4' : ['min','max','mean'],
         'c2': lambda x: (min(x)+0.0001)/max(max(x),0.0001)
    }
     
)
df_tmp_agg

c1   c4                        c2
      min   max  mean      <lambda>
0  0  0.0   0.0   0.0  1.000000e+00
1  1  0.0   7.0   3.5  1.000000e-06
2  2  0.0  14.0   7.0  5.000000e-07
3  3  0.0  21.0  10.5  3.333333e-07
4  4  0.0  28.0  14.0  2.500000e-07

In [ ]:
df_tmp_agg.columns

MultiIndex([('c1',         ''),
            ('c4',      'min'),
            ('c4',      'max'),
            ('c4',     'mean'),
            ('c2', '<lambda>')],
           )

In [ ]:
df_tmp_agg[('c2', '<lambda>')]

0    1.000000e+00
1    1.000000e-06
2    5.000000e-07
3    3.333333e-07
4    2.500000e-07
Name: (c2, <lambda>), dtype: float64

Putem indica pentru fiecare operație numele coloanei rezultat:

In [ ]:
df_tmp.groupby(['c1'], as_index = False).agg(

    c4_min = ('c4' ,'min'),
    c4_max = ('c4','max'),
    c4_mmean = ('c4','mean'),
    c2_special =('c2', lambda x: (min(x)+0.0001)/max(max(x),0.0001))
     
)

,c1,c4_min,c4_max,c4_mmean,c2_special
0,0,0.0,0.0,0.0,1.000000e+00
1,1,0.0,7.0,3.5,1.000000e-06
2,2,0.0,14.0,7.0,5.000000e-07
3,3,0.0,21.0,10.5,3.333333e-07
4,4,0.0,28.0,14.0,2.500000e-07


#### .transform

Uneori este nevoie de derivarea unor atribute la nivel de grup sub forma unor coloane care să rămână în tabela inițială. Aceasta se poate face cu singură intrucțiune care operează pe obiectele groupby transformând datele prin adăugarea coloanei dorite.



In [ ]:
dftmp1=df_tmp
df_tmp['c2_MedieGrup'] = dftmp1.groupby(['c1'], as_index = False).c2.transform('mean')
df_tmp

,c1,c2,c4,c2_MedieGrup
0,0,0.0,0.0,0.0
1,1,100.0,0.0,50.0
2,2,200.0,0.0,100.0
3,3,300.0,0.0,150.0
4,4,400.0,0.0,200.0
0,0,0.0,0.0,0.0
1,1,0.0,7.0,50.0
2,2,0.0,14.0,100.0
3,3,0.0,21.0,150.0
4,4,0.0,28.0,200.0


#### .pivot

Pivotarea datelor probabil că ați întâlnit-o la analiza datelor cu MS Excel. Acum pivotarea este posibilă, foarte similar cu ce se întâmplă în Excel, și în Google Sheets.

În Pandas pivotarea este o combinție între groupby și o funcție, pivot, care rearanjează datele. Iată un exemplu pe datele de mai sus:

In [ ]:
df_ex_pivot = df_tmp.assign(c0 = [0]*5 + [1]*5,c3 = df_tmp.c2+df_tmp.c4)[['c0','c1','c3']]
df_ex_pivot

,c0,c1,c3
0,0,0,0.0
1,0,1,100.0
2,0,2,200.0
3,0,3,300.0
4,0,4,400.0
0,1,0,0.0
1,1,1,7.0
2,1,2,14.0
3,1,3,21.0
4,1,4,28.0


In [ ]:
df_ex_pivot.pivot( index = 'c0', columns ='c1',values ='c3').reset_index(drop=False)

c1,c0,0,1,2,3,4
0,0,0.0,100.0,200.0,300.0,400.0
1,1,0.0,7.0,14.0,21.0,28.0


# Analiza exploratorie


In [ ]:
# Citirea datelor de laborator: nu trebuie modificată nicio cale.
# incarca("lab-set4") descarcă setul (doar prima dată) și întoarce tabelele lui;
# un tabel se alege după nume sau după poziție: date["OnlineRetail"] sau date[0].
from seed.date import incarca, lista_seturi
lista_seturi("lab")

## Detectare/eliminare anomalii 

Ne referim aici la valori care pot fi detectate de exemplu la o inspecție vizuală. De obicei le putem identifica folosind percentile ale distribuției valorilor. Există o ramură a învățării automate care se ocupă cu modele învățate să identifice anomalii în date, cum ar fi furtul de identitate și tranzacții suspecte la cardul de credit, sau valori ieșite din comun într-o serie temporală. Acest tip de modele depășește limitele acestui curs.

In [ ]:
date_echipa = incarca("lab-set7")
df7 = date_echipa[6][['InscClaimAmtReimbursed','DeductibleAmtPaid']]
df7.describe()


citire pt  Set7
Test_Beneficiarydata-1542969243754.csv
Test_Inpatientdata-1542969243754.csv
Test_Outpatientdata-1542969243754.csv
Test-1542969243754.csv
Train_Beneficiarydata-1542865627584.csv
Train_Inpatientdata-1542865627584.csv
Train_Outpatientdata-1542865627584.csv
Train-1542865627584.csv
setul de date  Set7  are  8 dataframe-uri.


,InscClaimAmtReimbursed,DeductibleAmtPaid
count,517737.000000,517737.000000
mean,286.334799,2.779233
std,694.034343,15.785839
min,0.000000,0.000000
25%,40.000000,0.000000
50%,80.000000,0.000000
75%,200.000000,0.000000
max,102500.000000,897.000000


După cum vedem ambele variabile ( InscClaimAmtReimbursed = valoare rambursată de compania de asigurări de sănătate, DeductibleAmtPaid = coplata ) prezintă o ciudățenie: diferență mare între a 75-a percentilă și maxim. Asta poate indica posibile încerări de fraudare și ar trebui studiate separat dacă de exemplu studiem valoarea medie rambursată în funcție de coplată.

Să vedem cum arată cuantilele de la 90 la max.

In [ ]:

df7.describe(percentiles =  [0.25,0.5,0.7,0.75, 0.8,0.85,0.9,.91,.92,.93,.94,.95,.96,.97,.98,.99])
        
    

,InscClaimAmtReimbursed,DeductibleAmtPaid
count,517737.000000,517737.000000
mean,286.334799,2.779233
std,694.034343,15.785839
min,0.000000,0.000000
25%,40.000000,0.000000
50%,80.000000,0.000000
70%,200.000000,0.000000
75%,200.000000,0.000000
80%,300.000000,0.000000
85%,500.000000,0.000000


In [ ]:
df7.sort_values(by = df7.columns[0], ascending = False).head(10)

,InscClaimAmtReimbursed,DeductibleAmtPaid
9326,102500,876
505756,101250,865
349263,95580,865
232849,85680,897
341672,84660,886
50435,84460,876
418824,83160,897
237553,17300,0
317240,17300,0
227767,17300,0


Izolăm excepțiile și le filtrăm.

In [ ]:
df7_exceptions = df7[df7.InscClaimAmtReimbursed > df7.InscClaimAmtReimbursed.quantile(0.99)]
display(df7_exceptions)
df7_no_exceptions = df7[df7.InscClaimAmtReimbursed <= df7.InscClaimAmtReimbursed.quantile(0.99)]
df7_no_exceptions.describe()

,InscClaimAmtReimbursed,DeductibleAmtPaid
755,9020,0
926,8060,0
2511,3400,0
2948,4100,0
2952,5700,0
...,...,...
517081,9100,0
517218,6020,0
517277,7600,0
517587,6070,0


,InscClaimAmtReimbursed,DeductibleAmtPaid
count,517182.000000,517182.000000
mean,278.090073,2.767865
std,556.479313,15.454588
min,0.000000,0.000000
25%,40.000000,0.000000
50%,80.000000,0.000000
75%,200.000000,0.000000
max,3300.000000,200.000000


## Imputare de valori

### Imputare cu funcții pandas

Cea mai simplă strategie este imputarea cu o valoare specificată.

Umplere cu valoarea anterioară:

- `ffil`

<b>Discuție</b> Ce valori rămân neacoperite?

Construim un exemplu pornind de la df-urile de mai sus.

In [ ]:
# df78 are goluri la sfârșit, dacă reordonăm pune ceva și la început. Apoi df_34 are goluri deci avem ceva goluri și la mijloc
def genereaza_date_exemplu():
    df_out = pd.concat([df78.sort_values(by = 'c3',ascending = False),df_34,df78])[['c1','c3']]\
        .reset_index()\
        .drop(columns = ['index'])
    # mai umplem din golurile de la mijloc
    df_out.loc[[8,10,12,14],'c3'] = df_out.loc[[8,10,12,14],'c1'] 
    return df_out

df_exemplu = genereaza_date_exemplu()

`info point` Implicit toate atribuirile din Python sunt la adresă (by reference). Asta înseamnă că modificare de valori trebuie făcută cu atenție. Voi ilustra acest lucru mai jos. Apoi vom folosi `copy.deepcopy` pentru a crea fizic o copie a obiectului dorit, cu alocare de memorie. 

Facem o 'copie' a exemplului și imputăm valorile lipsă din coloana c3 cu valoarea anterioară:

In [ ]:
# facem o 'copie'
df_ffill = df_exemplu

# facem imputarea co
df_ffill['c3'] = df_ffill['c3'].ffill()
df_ffill

,c1,c3
0,NaN,36.0
1,NaN,25.0
2,4.0,16.0
3,3.0,9.0
4,2.0,4.0
5,1.0,1.0
6,0.0,0.0
7,0.0,0.0
8,1.0,1.0
9,2.0,1.0


Ne uităm la original și constatăm că s-a modificat! Deci copia a fost la adresă (by reference...)

In [ ]:

df_exemplu

,c1,c3
0,NaN,36.0
1,NaN,25.0
2,4.0,16.0
3,3.0,9.0
4,2.0,4.0
5,1.0,1.0
6,0.0,0.0
7,0.0,0.0
8,1.0,1.0
9,2.0,1.0


Acum facem o copie profundă, care să și aloce altă zonă de memorie și refacem imputarea cu inspectarea ulterioară a celor două df-uri.

Pentru a mai acoperi ceva, de data asta facem umplerea tuturor coloanelor dintr-un foc:

In [ ]:
import copy
df_exemplu = genereaza_date_exemplu()

df_ffill_deep = copy.deepcopy(df_exemplu)
df_ffill_deep = df_ffill_deep.ffill()
print('imputarea')
display(df_ffill_deep)
print('originalul')
df_exemplu

imputarea


,c1,c3
0,NaN,36.0
1,NaN,25.0
2,4.0,16.0
3,3.0,9.0
4,2.0,4.0
5,1.0,1.0
6,0.0,0.0
7,0.0,0.0
8,1.0,1.0
9,2.0,1.0


originalul


,c1,c3
0,NaN,36.0
1,NaN,25.0
2,4.0,16.0
3,3.0,9.0
4,2.0,4.0
5,1.0,1.0
6,0.0,0.0
7,0.0,NaN
8,1.0,1.0
9,2.0,NaN


Umplere cu valoarea posterioară:
- `bfill`

<b>Discuție</b> Ce valori rămân neacoperite? Cum putem combina cele două metode pentru a umple complet valorile lipsă? Câte variante sunt?

In [ ]:
df_exemplu = genereaza_date_exemplu()

df_bfill = df_exemplu.bfill()
df_bfill

,c1,c3
0,4.0,36.0
1,4.0,25.0
2,4.0,16.0
3,3.0,9.0
4,2.0,4.0
5,1.0,1.0
6,0.0,0.0
7,0.0,1.0
8,1.0,1.0
9,2.0,3.0


In [ ]:
df_exemplu = genereaza_date_exemplu()

df_ffill_bfill = df_exemplu.ffill().bfill()
df_bfill_ffill = df_exemplu.bfill().ffill()

display(df_ffill_bfill)
display(df_bfill_ffill)

,c1,c3
0,4.0,36.0
1,4.0,25.0
2,4.0,16.0
3,3.0,9.0
4,2.0,4.0
5,1.0,1.0
6,0.0,0.0
7,0.0,0.0
8,1.0,1.0
9,2.0,1.0


,c1,c3
0,4.0,36.0
1,4.0,25.0
2,4.0,16.0
3,3.0,9.0
4,2.0,4.0
5,1.0,1.0
6,0.0,0.0
7,0.0,1.0
8,1.0,1.0
9,2.0,3.0


Umplere cu o valoare specificată ca și parametru:
- `fillna`

In [ ]:
df_exemplu.fillna(-1)

,c1,c3
0,-1.0,36.0
1,-1.0,25.0
2,4.0,16.0
3,3.0,9.0
4,2.0,4.0
5,1.0,1.0
6,0.0,0.0
7,0.0,-1.0
8,1.0,1.0
9,2.0,-1.0


Dacă vrem umplerea fiecărei coloane cu o statistică la nivelul coloanei, de exemplu media sau mediana valorilor existente:

In [ ]:
df_exemplu = genereaza_date_exemplu()
df_exemplu['c1'] = df_exemplu['c1'].fillna(df_exemplu['c1'].mean()) # umplere cu media
df_exemplu['c3'] = df_exemplu['c3'].fillna(df_exemplu['c3'].quantile(0.5)) # umplere cu mediana
df_exemplu

,c1,c3
0,2.0,36.0
1,2.0,25.0
2,4.0,16.0
3,3.0,9.0
4,2.0,4.0
5,1.0,1.0
6,0.0,0.0
7,0.0,4.0
8,1.0,1.0
9,2.0,4.0


Umplere prin interpolarea între valori anterioare și posterioare
- `interpolate`

Are sens doar pentru valori care pot fi interpolate, adică există valoare (valori) anteriore și posterioare înregistrării pe care dorim să o interpolăm.

Sunt multiple variante de interpolare disponibile pentru <a href = "https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.interpolate.html">pandas.interpolate</a>. Implicită este uzuala metodă de interpolare liniară. Puteți explora individual alte metode disponibile.

In [ ]:
df_exemplu = genereaza_date_exemplu()
df_umplere_cu_interpoalare_liniara = df_exemplu.interpolate()
df_umplere_cu_interpoalare_liniara

,c1,c3
0,NaN,36.000000
1,NaN,25.000000
2,4.0,16.000000
3,3.0,9.000000
4,2.0,4.000000
5,1.0,1.000000
6,0.0,0.000000
7,0.0,0.500000
8,1.0,1.000000
9,2.0,2.000000


### Imputare cu algoritmi

Pentru imputări mai sofisticate vă recomand <a href = "https://scikit-learn.org/stable/modules/impute.html">acest site</a>

In [ ]:
import random
df_exemplu = pd.DataFrame([
    { 'etichetă_clasă': 0 if random.random() < 0.1 else 1 ,
     'atribut1': random.random(),
     'atribut2': random.random()} for i in range(1000)
])
df_exemplu.groupby('etichetă_clasă').count()

,atribut1,atribut2
etichetă_clasă,,
0,116,116
1,884,884


### Balansarea claselor
#### Undersampling

Exemplu cu random undersampler(mai multe <a href = "https://machinelearningmastery.com/random-oversampling-and-undersampling-for-imbalanced-classification/">aici</a>):

In [ ]:
from imblearn.under_sampling import RandomUnderSampler

rnd_undersample = RandomUnderSampler(sampling_strategy='majority') #100/100; sau sampling_strategy=0.5 (200/100)
X,y = df_exemplu[['atribut1','atribut2']], df_exemplu['etichetă_clasă']
X_over, y_over = rnd_undersample.fit_resample(X, y)

df_undersample = pd.concat([y_over,X_over],axis=1)
df_undersample.groupby('etichetă_clasă').count()

In [ ]:
rnd_undersample = RandomUnderSampler(sampling_strategy=.5) #100/100; sau sampling_strategy=0.5 (200/100)
X,y = df_exemplu[['atribut1','atribut2']], df_exemplu['etichetă_clasă']
X_over, y_over = rnd_undersample.fit_resample(X, y)

df_undersample = pd.concat([y_over,X_over],axis=1)
df_undersample.groupby('etichetă_clasă').count()

,atribut1,atribut2
etichetă_clasă,,
0,116,116
1,232,232


#### Oversampling
Exemplu cu random oversampler

In [ ]:
from imblearn.over_sampling import RandomOverSampler

for r in [0.2,0.5,0.8,1]:
    print('pentru un raport între clase cu clasa mica în proporție de ',r)
    rnd_oversample = RandomOverSampler(sampling_strategy = r) 
    X,y = df_exemplu[['atribut1','atribut2']], df_exemplu['etichetă_clasă']
    X_over, y_over = rnd_oversample.fit_resample(X, y)

    df_oversample = pd.concat([y_over,X_over],axis=1)
    display(df_oversample.groupby('etichetă_clasă').count())

pentru un raport între clase cu clasa mica în proporție de  0.2


,atribut1,atribut2
etichetă_clasă,,
0,176,176
1,884,884


pentru un raport între clase cu clasa mica în proporție de  0.5


,atribut1,atribut2
etichetă_clasă,,
0,442,442
1,884,884


pentru un raport între clase cu clasa mica în proporție de  0.8


,atribut1,atribut2
etichetă_clasă,,
0,707,707
1,884,884


pentru un raport între clase cu clasa mica în proporție de  1


,atribut1,atribut2
etichetă_clasă,,
0,884,884
1,884,884


### Discretizare cu coloană de referință

Generăm un exemplu sintetic:

In [ ]:
df_exemplu = genereaza_date_exemplu().ffill().bfill()
df_exemplu['c1'] =round(df_exemplu['c1'].apply(lambda x: (x*random.random() + random.random()) * 10000),0)
df_exemplu= df_exemplu.rename(columns ={
    'c1': 'Total_vănzări',
    'c3': 'Ore_suplimentare'
})

print('Situație echipă vânzări')
df_exemplu

Situație echipă vânzări


,Total_vănzări,Ore_suplimentare
0,30294.0,36.0
1,25749.0,25.0
2,41981.0,16.0
3,14152.0,9.0
4,11237.0,4.0
5,8999.0,1.0
6,4697.0,0.0
7,8236.0,0.0
8,10803.0,1.0
9,19218.0,1.0


Vrem să grupăm agenții de vânzări în 5 grupe după numârul de ore suplimentare. Putem folosi venitu ca și coloană de referință pentru a nu introduce praguri arbitrare:

In [ ]:
# sortare după coloana pe care vrem să facem grupurile
df_exemplu = df_exemplu.sort_values(by = 'Ore_suplimentare')

# calcul procent cumulativ procentual din coloana valorilor de referință
total_vanzari = df_exemplu['Total_vănzări'].sum()
df_exemplu = df_exemplu.assign( vanz_proc_cum = round(df_exemplu['Total_vănzări'].cumsum()/total_vanzari,2))
df_exemplu

,Total_vănzări,Ore_suplimentare,vanz_proc_cum
17,9197.0,0.0,0.02
6,4697.0,0.0,0.03
7,8236.0,0.0,0.05
13,8235.0,0.0,0.07
12,1832.0,0.0,0.08
18,12963.0,1.0,0.11
5,8999.0,1.0,0.13
8,10803.0,1.0,0.15
9,19218.0,1.0,0.20
16,32003.0,2.0,0.27


In [ ]:
limite_intervale = [ df_exemplu[df_exemplu['vanz_proc_cum'] < lim]['Ore_suplimentare'].max()
                             for lim in [0.2, 0.4, 0.6, 0.8] ]
limite_intervale

[1.0, 2.0, 9.0, 16.0]

In [ ]:
limite_intervale = [ df_exemplu[df_exemplu['vanz_proc_cum'] < lim]['Ore_suplimentare'].max()
                             for lim in [0.33, 0.67] ]
limite_intervale

[2.0, 9.0]

In [ ]:


df_exemplu['Grupare'] = df_exemplu['Ore_suplimentare']\
        .apply(lambda x: 'G1' if x <= limite_intervale[0] else
                          'G2' if x <= limite_intervale[1] else
                          'G3')
df_exemplu

,Total_vănzări,Ore_suplimentare,vanz_proc_cum,Grupare
17,9197.0,0.0,0.02,G1
6,4697.0,0.0,0.03,G1
7,8236.0,0.0,0.05,G1
13,8235.0,0.0,0.07,G1
12,1832.0,0.0,0.08,G1
18,12963.0,1.0,0.11,G1
5,8999.0,1.0,0.13,G1
8,10803.0,1.0,0.15,G1
9,19218.0,1.0,0.20,G1
16,32003.0,2.0,0.27,G1


### Standardizare, scalare


#### Exemplu manual de standardizare

In [ ]:
df_exemplu.info()

<class 'pandas.core.frame.DataFrame'>
Index: 24 entries, 17 to 23
Data columns (total 4 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Total_vănzări     24 non-null     float64
 1   Ore_suplimentare  24 non-null     float64
 2   vanz_proc_cum     24 non-null     float64
 3   Grupare           24 non-null     object 
dtypes: float64(3), object(1)
memory usage: 960.0+ bytes


In [ ]:
# colectăm mediile.min și max pe fiecare coloană{
col_stats = {
        c: {
            'min':df_exemplu[c].min(),
            'max':df_exemplu[c].max(),
            'medie':df_exemplu[c].mean(),
            'devstd':df_exemplu[c].std()
        }
    for c in ['Total_vănzări','Ore_suplimentare'  ]
}

for c in ['Total_vănzări','Ore_suplimentare'  ]:
    
    # normalizare/ standardizare Z-score
    df_exemplu [ c+'_std_manual' ]= (df_exemplu[c] - col_stats[c]['medie'])/col_stats[c]['devstd']
    
    # scalare de tip min-max
    df_exemplu [ c+'_minMax_manual' ]= (df_exemplu[c] - col_stats[c]['min'])/(col_stats[c]['max']-col_stats[c]['min'])
    
df_exemplu.describe().T

,count,mean,std,min,25%,50%,75%,max
Total_vănzări,24.0,1.783062e+04,11216.132476,1832.000000,8808.250000,13557.500000,27758.250000,41981.000000
Ore_suplimentare,24.0,8.166667e+00,11.385218,0.000000,1.000000,2.500000,10.750000,36.000000
vanz_proc_cum,24.0,4.004167e-01,0.305905,0.020000,0.125000,0.365000,0.605000,1.000000
Total_vănzări_std_manual,24.0,2.312965e-18,1.000000,-1.426394,-0.804411,-0.380980,0.885120,2.153182
Total_vănzări_minMax_manual,24.0,3.984813e-01,0.279363,0.000000,0.173759,0.292050,0.645751,1.000000
Ore_suplimentare_std_manual,24.0,1.110223e-16,1.000000,-0.717304,-0.629471,-0.497721,0.226902,2.444690
Ore_suplimentare_minMax_manual,24.0,2.268519e-01,0.316256,0.000000,0.027778,0.069444,0.298611,1.000000


`Discuție` Ce se întâmplâ când actualizăm datele? În practica învățării automate, datele se împart în date de antrenament, adesea numite și date istorice și date de test, care joacă rolul unor date noi, neaccesibile la antrenament. Atât datele noi cât și datele de antrenament trebuie scalate. Cum procedăm?

#### Standardizare, scalare cu metode sklearn

Fit, apoi transform:

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

std_scaler = StandardScaler()
minmax_scaler = MinMaxScaler()

df_exemplu_std_scaler = copy.deepcopy(df_exemplu[['Total_vănzări','Ore_suplimentare' ]])

for c in ['Total_vănzări','Ore_suplimentare' ]:
    std_scaler.fit(df_exemplu_std_scaler[[c]])
    minmax_scaler.fit(df_exemplu_std_scaler[[c]])
    df_exemplu_std_scaler[c+'_std_scaler'] = std_scaler.transform(df_exemplu_std_scaler[[c]])
    df_exemplu_std_scaler[c+'_minMax_scaler'] = minmax_scaler.transform(df_exemplu_std_scaler[[c]])
df_exemplu_std_scaler.describe().T

,count,mean,std,min,25%,50%,75%,max
Total_vănzări,24.0,1.783062e+04,11216.132476,1832.000000,8808.250000,13557.500000,27758.250000,41981.000000
Ore_suplimentare,24.0,8.166667e+00,11.385218,0.000000,1.000000,2.500000,10.750000,36.000000
Total_vănzări_std_scaler,24.0,-9.251859e-18,1.021508,-1.457073,-0.821712,-0.389174,0.904157,2.199492
Total_vănzări_minMax_scaler,24.0,3.984813e-01,0.279363,0.000000,0.173759,0.292050,0.645751,1.000000
Ore_suplimentare_std_scaler,24.0,5.551115e-17,1.021508,-0.732732,-0.643010,-0.508426,0.231783,2.497270
Ore_suplimentare_minMax_scaler,24.0,2.268519e-01,0.316256,0.000000,0.027778,0.069444,0.298611,1.000000


Fit_transform - dacă transformarea se face tot pe datele pe care s-a inițializat obiectul:

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

std_scaler = StandardScaler()
minmax_scaler = MinMaxScaler()

df_exemplu_std_scaler = copy.deepcopy(df_exemplu[['Total_vănzări','Ore_suplimentare' ]])

for c in ['Total_vănzări','Ore_suplimentare' ]:
    df_exemplu_std_scaler[c+'_std_scaler'] = std_scaler.fit_transform(df_exemplu_std_scaler[[c]])
    df_exemplu_std_scaler[c+'_minMax_scaler'] = minmax_scaler.fit_transform(df_exemplu_std_scaler[[c]])
df_exemplu_std_scaler.describe().T

,count,mean,std,min,25%,50%,75%,max
Total_vănzări,24.0,1.783062e+04,11216.132476,1832.000000,8808.250000,13557.500000,27758.250000,41981.000000
Ore_suplimentare,24.0,8.166667e+00,11.385218,0.000000,1.000000,2.500000,10.750000,36.000000
Total_vănzări_std_scaler,24.0,-9.251859e-18,1.021508,-1.457073,-0.821712,-0.389174,0.904157,2.199492
Total_vănzări_minMax_scaler,24.0,3.984813e-01,0.279363,0.000000,0.173759,0.292050,0.645751,1.000000
Ore_suplimentare_std_scaler,24.0,5.551115e-17,1.021508,-0.732732,-0.643010,-0.508426,0.231783,2.497270
Ore_suplimentare_minMax_scaler,24.0,2.268519e-01,0.316256,0.000000,0.027778,0.069444,0.298611,1.000000


`Discuție` Impactul datelor noi - fit, fit-transform, transform

### Codificare de tip One-Hot-Encoding

#### Metoda pandas - get_dummies

In [ ]:
df_exemplu_OHE1= copy.deepcopy(df_exemplu[['Total_vănzări','Ore_suplimentare','vanz_proc_cum','Grupare']])

display(df_exemplu_OHE1)

# apelăm get_dummies - metoda pandas pentru OHE
df_exemplu_OHE1_1 = pd.get_dummies(df_exemplu_OHE1, columns = ['Grupare'])

display(df_exemplu_OHE1_1)

# precizarea unui prefix și a datelor ca numere reale
df_exemplu_OHE1_2 = pd.get_dummies(df_exemplu_OHE1, columns = ['Grupare'], prefix = 'Clasa',dtype = float)
df_exemplu_OHE1_2

,Total_vănzări,Ore_suplimentare,vanz_proc_cum,Grupare
17,9197.0,0.0,0.02,G1
6,4697.0,0.0,0.03,G1
7,8236.0,0.0,0.05,G1
13,8235.0,0.0,0.07,G1
12,1832.0,0.0,0.08,G1
18,12963.0,1.0,0.11,G1
5,8999.0,1.0,0.13,G1
8,10803.0,1.0,0.15,G1
9,19218.0,1.0,0.20,G1
16,32003.0,2.0,0.27,G1


,Total_vănzări,Ore_suplimentare,vanz_proc_cum,Grupare_G1,Grupare_G2,Grupare_G3
17,9197.0,0.0,0.02,True,False,False
6,4697.0,0.0,0.03,True,False,False
7,8236.0,0.0,0.05,True,False,False
13,8235.0,0.0,0.07,True,False,False
12,1832.0,0.0,0.08,True,False,False
18,12963.0,1.0,0.11,True,False,False
5,8999.0,1.0,0.13,True,False,False
8,10803.0,1.0,0.15,True,False,False
9,19218.0,1.0,0.20,True,False,False
16,32003.0,2.0,0.27,True,False,False


,Total_vănzări,Ore_suplimentare,vanz_proc_cum,Clasa_G1,Clasa_G2,Clasa_G3
17,9197.0,0.0,0.02,1.0,0.0,0.0
6,4697.0,0.0,0.03,1.0,0.0,0.0
7,8236.0,0.0,0.05,1.0,0.0,0.0
13,8235.0,0.0,0.07,1.0,0.0,0.0
12,1832.0,0.0,0.08,1.0,0.0,0.0
18,12963.0,1.0,0.11,1.0,0.0,0.0
5,8999.0,1.0,0.13,1.0,0.0,0.0
8,10803.0,1.0,0.15,1.0,0.0,0.0
9,19218.0,1.0,0.20,1.0,0.0,0.0
16,32003.0,2.0,0.27,1.0,0.0,0.0


#### Metoda <a href ="https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html">sklearn</a>

In [ ]:
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder()

df_exemplu_OHE1= copy.deepcopy(df_exemplu[['Total_vănzări','Ore_suplimentare','vanz_proc_cum','Grupare']])


enc_data = pd.DataFrame(ohe.fit_transform( 
    df_exemplu[['Grupare']]).toarray()) 
  
# Merge with main 
df_exemplu_OHE1_3 = df_exemplu_OHE1.join(enc_data) 
df_exemplu_OHE1_3

,Total_vănzări,Ore_suplimentare,vanz_proc_cum,Grupare,0,1,2
17,9197.0,0.0,0.02,G1,0.0,1.0,0.0
6,4697.0,0.0,0.03,G1,1.0,0.0,0.0
7,8236.0,0.0,0.05,G1,1.0,0.0,0.0
13,8235.0,0.0,0.07,G1,0.0,1.0,0.0
12,1832.0,0.0,0.08,G1,0.0,1.0,0.0
18,12963.0,1.0,0.11,G1,0.0,0.0,1.0
5,8999.0,1.0,0.13,G1,1.0,0.0,0.0
8,10803.0,1.0,0.15,G1,1.0,0.0,0.0
9,19218.0,1.0,0.20,G1,1.0,0.0,0.0
16,32003.0,2.0,0.27,G1,0.0,1.0,0.0


`info point` Get_dummies este mai transparentă. În schimb OneHotEncoder poate fi încorporată într-o linie de asamblare, ceea ce poate fi foarte important în practică dacă lucrăm cu date și modele multe.